# 04 - Segmentation, profitability and retention prioritisation

Turns the model outputs into business answers: who are the customers, who makes money, where is the risk,
and who should the retention team contact?

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from src.db import read_sql
from src.evaluation import plots  # applies the shared chart style
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
FIG = Path("../reports/figures")
R = json.load(open("../reports/model_results.json"))
q = lambda name: read_sql(open(f"../sql/analysis/{name}.sql").read())

## 1. Segmentation
K-means on six behaviour ratios (utilization, share of bill paid, share of months paid in full, share of months
inactive, new charges / limit, share of months 60+ DPD). k was chosen with silhouette and bootstrap stability:

In [2]:
pd.DataFrame(R["segmentation"]["k_selection"])

,k,inertia,silhouette,stability_ari
0,2,"108,148.9057",0.4137,0.9966
1,3,"84,089.8142",0.4380,0.9975
2,4,"61,372.5564",0.4490,0.9987
3,5,"47,165.0712",0.4301,0.9848
4,6,"38,171.6086",0.4153,0.9890
5,7,"34,434.0562",0.3781,0.9930
6,8,"31,240.6602",0.3742,0.9104


k = 4 has the best silhouette and k = 5 is close; both are very stable (ARI ~0.99). I used k = 5 because it
splits out heavy-spend revolvers, whose economics differ from ordinary revolvers.

In [3]:
pd.DataFrame(R["segmentation"]["profile"])

,segment,customers,avg_utilization,avg_payment_ratio,share_months_paid_in_full,share_months_inactive,new_charges_to_limit,share_months_60plus,avg_credit_limit
0,Delinquent revolvers,3156,0.6107,0.2267,0.0089,0.0034,0.0301,0.7481,"94,017.7440"
1,Dormant / low use,3564,0.0251,0.9601,0.1622,0.7249,0.0196,0.0238,"197,836.7003"
2,Heavy-spend revolvers,4026,0.6073,0.3961,0.1025,0.0389,0.2053,0.0543,"97,302.5335"
3,Revolvers,12073,0.5293,0.2545,0.0151,0.0031,0.0479,0.0564,"163,811.7850"
4,Transactors,7181,0.0422,0.8985,0.7462,0.0647,0.0335,0.0286,"230,229.7730"


## 2. Profitability (estimates)

In [4]:
q("executive_kpis").T

,0
customers,"30,000.0000"
active_customers_sep,"27,606.0000"
total_balance_sep,"1,537,381,257.0000"
total_credit_limit,"5,024,529,680.0000"
portfolio_utilization,0.3060
est_monthly_purchase_volume,"219,027,147.4360"
est_monthly_revenue,"21,995,435.6945"
est_monthly_contribution_before_losses,"17,688,855.2294"
ecl_next_month,"10,249,208.3800"
est_monthly_risk_adj_contribution,"7,439,646.8494"


In [5]:
seg = q("segment_summary")
seg[["segment", "customers", "avg_monthly_purchases", "actual_default_rate", "avg_pd", "avg_dormancy_score",
     "total_monthly_revenue", "total_ecl", "total_risk_adj_contribution", "share_loss_making", "share_of_total_contribution"]]

,segment,customers,avg_monthly_purchases,actual_default_rate,avg_pd,avg_dormancy_score,total_monthly_revenue,total_ecl,total_risk_adj_contribution,share_loss_making,share_of_total_contribution
0,Revolvers,12073,"6,490.4854",0.1633,0.1657,0.0035,"13,562,089.0048","4,190,133.1032","7,123,934.4451",0.0911,0.9576
1,Heavy-spend revolvers,4026,"19,209.5882",0.2027,0.2015,0.0026,"3,882,351.5033","1,812,943.4592","1,142,601.5017",0.1617,0.1536
2,Transactors,7181,"6,826.9702",0.1439,0.1399,0.0275,"1,052,335.8818","665,884.5842","-176,817.6357",0.7222,-0.0238
3,Delinquent revolvers,3156,"1,794.3172",0.6280,0.6254,0.0050,"3,213,071.4756","3,013,997.9182","-199,481.4851",0.4072,-0.0268
4,Dormant / low use,3564,"2,424.9092",0.2340,0.2353,0.0707,"285,587.8291","566,249.3153","-450,589.9766",0.9004,-0.0606


Ordinary revolvers carry most of the estimated value: interest on revolving balances is the main revenue line.
Delinquent revolvers generate plenty of revenue but nearly all of it is offset by expected losses. Transactors
and dormant/low-use customers come out slightly negative because interchange on modest spend barely covers
servicing costs plus a share of expected losses. That last result depends on the assumptions (see sensitivity).

In [6]:
c = q("value_concentration")
c[c.top_percent_of_customers.isin([1, 5, 10, 20, 50, 60, 80, 100])]

,top_percent_of_customers,contribution,cum_contribution,cum_share_of_total
0,1,"974,813.0573","974,813.0573",0.1310
4,5,"479,572.0925","3,258,996.7369",0.4381
9,10,"314,754.9225","5,122,777.2995",0.6886
19,20,"154,519.9882","7,288,952.7742",0.9797
49,50,"26,627.7592","9,379,606.7703",1.2608
59,60,"4,755.5146","9,524,805.3017",1.2803
79,80,"-26,049.6544","9,278,109.8821",1.2471
99,100,"-535,758.3107","7,439,646.8494",1.0000


In [7]:
q("profit_sensitivity")

,annual_loss_rate,monthly_revenue,monthly_ecl,monthly_risk_adj_contribution,share_loss_making
0,0.0400,"21,995,435.6945","5,124,604.1900","12,564,251.0394",0.2441
1,0.0800,"21,995,435.6945","10,249,208.3800","7,439,646.8494",0.3810
2,0.1200,"21,995,435.6945","15,373,812.5700","2,315,042.6594",0.4872


The portfolio stays profitable across the 4-12% loss-rate range, but the share of loss-making customers moves
from about a quarter to about half, so customer-level profitability should be read as a ranking more than as
precise numbers.

## 3. Where is the credit risk?

In [8]:
q("risk_by_group")

,dimension,grp,customers,actual_default_rate,avg_pd,total_ecl,total_ead,ecl_rate
0,Age band,21-24,2685,0.2719,0.2726,"596,531.7782","88,977,075.3000",0.0067
1,Age band,25-34,13011,0.2030,0.2050,"4,239,147.9766","833,318,033.2000",0.0051
2,Age band,35-44,9018,0.2186,0.2155,"3,405,183.4030","620,593,836.1000",0.0055
3,Age band,45-54,4233,0.2393,0.2371,"1,578,895.7673","276,120,728.2000",0.0057
4,Age band,55+,1053,0.2669,0.2680,"429,449.4550","69,658,110.1000",0.0062
5,Credit limit tier,1. <50k,4311,0.3607,0.3577,"669,710.5389","69,316,983.8000",0.0097
6,Credit limit tier,2. 50k-150k,10979,0.2520,0.2512,"3,610,997.1460","525,237,515.2000",0.0069
7,Credit limit tier,3. 150k-300k,9593,0.1682,0.1700,"3,577,919.6404","718,817,570.1000",0.0050
8,Credit limit tier,4. 300k+,5117,0.1368,0.1364,"2,390,581.0548","575,295,713.8000",0.0042
9,Education,Graduate school,10585,0.1923,0.1915,"3,377,518.8766","691,436,607.2000",0.0049


## 4. Retention prioritisation

In [9]:
read_sql('''SELECT priority_tier, count(*) customers, avg(churn_score) avg_p_dormant, avg(pd_score) avg_pd,
                  avg(est_risk_adjusted_contribution) avg_value, sum(expected_value_at_risk) value_at_risk
           FROM mart.retention_priority GROUP BY 1 ORDER BY 1''')

,priority_tier,customers,avg_p_dormant,avg_pd,avg_value,value_at_risk
0,1. Contact now,57,0.0758,0.0928,"1,518.6573","56,003.0332"
1,3. Monitor - offer costs more than expected gain,16799,0.0041,0.1413,548.7267,"212,919.4859"
2,4. Exclude - not profitable,7082,0.0399,0.1661,-91.6475,0.0000
3,5. Exclude - high credit risk,3668,0.0051,0.6718,-262.0505,"13,156.8181"


In [10]:
read_sql('''SELECT segment, count(*) active, avg(churn_score) avg_p_dormant,
                  avg(est_risk_adjusted_contribution) avg_value, sum(expected_value_at_risk) value_at_risk
           FROM mart.retention_priority GROUP BY 1 ORDER BY value_at_risk DESC''')

,segment,active,avg_p_dormant,avg_value,value_at_risk
0,Revolvers,12031,0.0035,590.7677,"159,379.0093"
1,Heavy-spend revolvers,4008,0.0026,283.7148,"52,246.4926"
2,Transactors,6689,0.0275,-23.8201,"47,680.6045"
3,Delinquent revolvers,3154,0.0050,-63.3333,"16,615.7559"
4,Dormant / low use,1724,0.0707,-110.9023,"6,157.4750"


The customers most likely to go dormant (dormant/low-use and transactors) are the ones with little value,
while the valuable revolvers rarely go dormant. The total 12-month value at risk from dormancy (about NT$0.3M)
is tiny next to one month of expected credit loss (about NT$10M), and an expensive retention offer only pays
for a few dozen customers:

In [11]:
q("retention_cost_sensitivity")

,contact_cost,save_rate,customers_worth_contacting,total_expected_net_benefit
0,10,0.1000,496,"9,355.2569"
1,10,0.2000,960,"25,642.5928"
2,50,0.2000,161,"10,221.6149"
3,100,0.2000,57,"5,500.6066"
4,100,0.3000,104,"12,076.9462"


In [12]:
b = q("retention_budget_curve")
b.loc[b.groupby("contact_cost").cum_benefit_model.idxmax()]

,contact_cost,customers_contacted,share_of_active_contacted,cum_benefit_model,cum_benefit_random
192,10,960,0.0348,"25,642.5928","-6,536.8104"
585,50,160,0.0058,"10,221.5229","-7,489.4684"
1118,100,60,0.0022,"5,499.0283","-5,808.5507"


With a 10% budget (about 2,450 customers) the budget is not the constraint: at NT$100 per contact the best
plan contacts only ~50-60 customers, and even a NT$10 digital nudge is worth sending to under 1,000. Contacting
more people in model order starts to lose money.

**Caveats:** the model predicts who is likely to go dormant, not who will respond to an offer, and the save rate
is an assumption. Any campaign should keep a random holdout group so the actual uplift can be measured.

In [13]:
read_sql('''SELECT customer_id, segment, round(churn_score::numeric, 3) p_dormant, round(pd_score::numeric, 3) pd,
                  round(est_risk_adjusted_contribution::numeric) value_per_month,
                  round(expected_net_benefit::numeric) net_benefit, priority_reason
           FROM mart.retention_priority WHERE priority_tier = '1. Contact now' ORDER BY priority_rank LIMIT 10''')

,customer_id,segment,p_dormant,pd,value_per_month,net_benefit,priority_reason
0,5297,Heavy-spend revolvers,0.0980,0.1000,"3,026.0000",609.0000,P(dormant) 9.8% | PD 10% | risk-adj. value NT$...
1,8501,Revolvers,0.0670,0.1570,"3,970.0000",535.0000,P(dormant) 6.7% | PD 16% | risk-adj. value NT$...
2,25950,Heavy-spend revolvers,0.1230,0.0790,"1,806.0000",433.0000,P(dormant) 12.3% | PD 8% | risk-adj. value NT$...
3,20963,Revolvers,0.2200,0.2770,818.0000,332.0000,P(dormant) 22.0% | PD 28% | risk-adj. value NT...
4,8323,Transactors,0.1350,0.0780,"1,107.0000",258.0000,P(dormant) 13.5% | PD 8% | risk-adj. value NT$...
5,29004,Revolvers,0.0570,0.0910,"2,615.0000",258.0000,P(dormant) 5.7% | PD 9% | risk-adj. value NT$2...
6,28042,Revolvers,0.1010,0.0610,"1,433.0000",248.0000,P(dormant) 10.1% | PD 6% | risk-adj. value NT$...
7,24216,Revolvers,0.0840,0.0460,"1,702.0000",243.0000,P(dormant) 8.4% | PD 5% | risk-adj. value NT$1...
8,26950,Revolvers,0.0650,0.0690,"1,783.0000",176.0000,P(dormant) 6.5% | PD 7% | risk-adj. value NT$1...
9,22088,Revolvers,0.0490,0.0630,"2,215.0000",161.0000,P(dormant) 4.9% | PD 6% | risk-adj. value NT$2...


## 5. Are the segment differences statistically meaningful?

In [14]:
pd.DataFrame(json.load(open("../reports/business_summary.json"))["significance_tests"]).T

,chi2,dof,p_value
default_rate_by_segment,"3,527.9486",4.0000,0.0000
default_rate_by_limit_tier,915.3156,3.0000,0.0000
default_rate_by_sex,47.7088,1.0000,0.0000
dormancy_rate_by_segment_jul_snapshot,"5,854.9816",4.0000,0.0000


Default and dormancy rates differ across segments far more than chance would explain (very small p-values).
With 30,000 customers almost any difference is "significant", so the size of the gap matters more than the
p-value; the gaps here are large (e.g. ~63% vs ~14% default rate for delinquent revolvers vs transactors).